In [ ]:


# # 📘 Scenario: Corporate Expansion Analysis
# # Context:
# # You are part of a business intelligence team at a multinational company. Your manager has asked you to analyze press
# releases and news articles to identify key entities such as organizations, people, locations, dates, and numbers. This
#  information will help the company track competitors’ moves and plan strategy.
# # Task:
# # You receive the following press release excerpt:
# # "Apple Inc. plans a new office in Hyderabad, India. Tim Cook announced this in March 2023. The site will create 5,000 jobs
# and focus on innovative technologies in supply chain analytics."

# # Your job is to run Named Entity Recognition (NER) using spaCy to automatically extract structured insights.


import spacy

# Load the small English model
nlp = spacy.load("en_core_web_sm")

text = """
Apple Inc. plans a new office in Hyderabad, India. Tim Cook announced this in March 2023.
The site will create 5,000 jobs and focus on innovative technologies in supply chain analytics.
"""

# Load the spaCy NLP pipeline (make sure the model is installed)
# Example: python -m spacy download en_core_web_sm
import spacy

nlp = spacy.load("en_core_web_sm")

# Example text for entity extraction
text = """
Apple Inc. was founded by Steve Jobs and Steve Wozniak in California.
It is headquartered in Cupertino and has offices in New York and London.
In 2023, Apple reported revenue of $394 billion.
"""

# Process the text with spaCy NLP pipeline
# This performs tokenization, POS tagging, dependency parsing, and NER
doc = nlp(text)

# -------------------------------------------------------------
# Print all named entities detected in the text
# Each entity will show:
# 1. The actual entity text
# 2. The entity label (type of entity)
# -------------------------------------------------------------
print("Named Entities (text, label):")

for ent in doc.ents:
    # ent.text  -> the detected entity
    # ent.label_ -> the category of entity (ORG, PERSON, GPE etc.)
    print(f"{ent.text:25} -> {ent.label_}")

# -------------------------------------------------------------
# Optional Section: Group entities by their label
# This helps summarize entities under each category
# Example:
# PERSON: Steve Jobs, Steve Wozniak
# ORG: Apple
# GPE: California, New York
# -------------------------------------------------------------

from collections import defaultdict

# defaultdict automatically creates empty lists for new keys
by_label = defaultdict(list)

# Loop through entities and group them by label
for ent in doc.ents:
    by_label[ent.label_].append(ent.text)

# -------------------------------------------------------------
# Print grouped entities
# sorted(set(items)) removes duplicates and sorts results
# -------------------------------------------------------------
print("\nEntities grouped by label:")

for label, items in by_label.items():
    print(f"{label}: {sorted(set(items))}")

Named Entities (text, label):
Apple Inc.                -> ORG
Steve Jobs                -> PERSON
Steve Wozniak             -> PERSON
California                -> GPE
Cupertino                 -> GPE
New York                  -> GPE
London                    -> GPE
2023                      -> DATE
Apple                     -> ORG
$394 billion              -> MONEY

Entities grouped by label:
ORG: ['Apple', 'Apple Inc.']
PERSON: ['Steve Jobs', 'Steve Wozniak']
GPE: ['California', 'Cupertino', 'London', 'New York']
DATE: ['2023']
MONEY: ['$394 billion']


In [ ]:
# Word embeddings with gensim Word2Vec
# Setup: python -m pip install gensim nltk
!pip install gensim nltk

import nltk
nltk.download("punkt")
nltk.download("punkt_tab") # Added to resolve LookupError
from nltk.tokenize import sent_tokenize, word_tokenize

from gensim.models import Word2Vec

corpus = """
Manufacturing relies on predictive maintenance and supply chain optimization.
Data engineers build pipelines, while analysts monitor KPIs and anomalies.
Robotics and IoT sensors stream telemetry to cloud databases for real-time insights.
Quality control uses computer vision to detect defects on the shop floor.
"""

# Tokenize sentences -> words
sentences = [word_tokenize(s.lower()) for s in sent_tokenize(corpus)]

# Train a small Word2Vec model
model = Word2Vec(
    sentences,
    vector_size=50,   # embedding dimension
    window=5,         # context window size
    min_count=1,      # keep all words for demo
    workers=2,
    sg=1              # skip-gram; use 0 for CBOW
)

# Explore similar words
for target in ["maintenance", "supply", "quality", "telemetry"]:
    print(f"\nTop similar to '{target}':")
    try:
        for w, score in model.wv.most_similar(target, topn=5):
            print(f"{w:15} -> {score:.3f}")
    except KeyError:
        print("Word not in vocabulary.")

# Cosine similarity between pairs
from numpy import dot
from numpy.linalg import norm

def cosine(u, v):
    return dot(u, v) / (norm(u) * norm(v))

pairs = [("maintenance", "telemetry"),
         ("quality", "defects"),
         ("supply", "optimization")]

print("\nCosine similarities:")
for a, b in pairs:
    try:
        sim = cosine(model.wv[a], model.wv[b])
        print(f"{a:12} ~ {b:12} -> {sim:.3f}")
    except KeyError:
        print(f"Missing word: {a} or {b}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 59.5 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!



Top similar to 'maintenance':
real-time       -> 0.240
defects         -> 0.240
analysts        -> 0.236
kpis            -> 0.224
detect          -> 0.204

Top similar to 'supply':
optimization    -> 0.286
to              -> 0.252
cloud           -> 0.192
analysts        -> 0.169
maintenance     -> 0.169

Top similar to 'quality':
shop            -> 0.307
cloud           -> 0.267
uses            -> 0.191
floor           -> 0.154
.               -> 0.151

Top similar to 'telemetry':
predictive      -> 0.171
cloud           -> 0.171
manufacturing   -> 0.167
iot             -> 0.166
for             -> 0.164

Cosine similarities:
maintenance  ~ telemetry    -> -0.145
quality      ~ defects      -> 0.119
supply       ~ optimization -> 0.286


In [ ]:
# Code Demo (Trigram counts)
from collections import defaultdict

corpus = "the cat sat on the mat the cat lay on the rug the dog barked loudly"
tokens = corpus.split()

# Build trigram counts
trigram_counts = defaultdict(lambda: defaultdict(int))
for i in range(len(tokens)-2):
    context = (tokens[i], tokens[i+1])
    next_word = tokens[i+2]
    trigram_counts[context][next_word] += 1

# Predict next word after "the cat"
context = ("the", "cat")
print("Next word predictions for context:", context)
for word, count in trigram_counts[context].items():
    print(f"{word} -> {count}")

Next word predictions for context: ('the', 'cat')
sat -> 1
lay -> 1


In [ ]:
# Code Demo (Simple feedforward LM with Keras)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Flatten

# Vocabulary size and embedding dimension
vocab_size = 50
embed_dim = 8

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=2), # context length=2
    Flatten(),
    Dense(32, activation='relu'),
    Dense(vocab_size, activation='softmax')  # predict next word
])

model.compile(optimizer='adam', loss='categorical_crossentropy')
model.build(input_shape=(None, 2))  # batch size flexible, sequence length = 2

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 2, 8)           │           400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 50)             │         1,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,594 (10.13 KB)

 Trainable params: 2,594 (10.13 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#Code Demo (LSTM for next‑word prediction)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import numpy as np

# Vocabulary size and embedding dimension
vocab_size = 100
embed_dim = 16
seq_length = 5  # input sequence length

# Define the model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=seq_length),
    LSTM(64),  # LSTM layer with 64 units
    Dense(vocab_size, activation='softmax')  # predict next word
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy')

# Force the model to build by providing input shape
model.build(input_shape=(None, seq_length))

# Show the summary
model.summary()

# Optional: run a dummy prediction to confirm
dummy_input = np.random.randint(0, vocab_size, (1, seq_length))
print("Dummy prediction shape:", model.predict(dummy_input).shape)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 5, 16)          │         1,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        20,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │         6,500 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,836 (112.64 KB)

 Trainable params: 28,836 (112.64 KB)

 Non-trainable params: 0 (0.00 B)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 455ms/step
Dummy prediction shape: (1, 100)


In [ ]:
from transformers import pipeline

# Load the sentiment analysis pipeline
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Input text
text = "I absolutely love this movie! Great acting."

# Perform classification
result = classifier(text)

# Display result
print(result)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998805522918701}]


In [ ]:
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense
import numpy as np # Import numpy for dummy data generation

# Define vocabulary size and sequence length
vocab_size = 100
seq_len = 5

# Define model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128), # Removed input_length as it's deprecated
    LSTM(256, return_sequences=True),
    LSTM(256),
    Dense(vocab_size, activation='softmax')
])

# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # use categorical_crossentropy if y_train is one-hot
    metrics=['accuracy']
)

# Generate dummy data for demonstration
num_samples = 100 # Number of dummy samples
X_train = np.random.randint(0, vocab_size, (num_samples, seq_len))
y_train = np.random.randint(0, vocab_size, (num_samples,))

# Train model
history = model.fit(
    X_train,
    y_train,
    epochs=2, # Reduced epochs for a quick demo run
    batch_size=64,          # adjust based on dataset size
    validation_split=0.2,   # or use validation_data=(X_val, y_val)
    verbose=1
)

Epoch 1/2
2/2 ━━━━━━━━━━━━━━━━━━━━ 11s 578ms/step - accuracy: 0.0000e+00 - loss: 4.6048 - val_accuracy: 0.0000e+00 - val_loss: 4.6058
Epoch 2/2
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.2198 - loss: 4.5946 - val_accuracy: 0.0000e+00 - val_loss: 4.6056


In [ ]:
import spacy

# Load the small English model
nlp = spacy.load("en_core_web_sm")

# Sports match report text for entity extraction
text = """
Manchester United defeated Real Madrid 3-2 at Wembley Stadium, London. Cristiano Ronaldo scored twice, while Marcus Rashford netted the winning goal in July 2024.
"""

# Process the text with spaCy NLP pipeline
doc = nlp(text)

print("Named Entities (text, label):")
for ent in doc.ents:
    print(f"{ent.text:25} -> {ent.label_}")

Named Entities (text, label):
Manchester United         -> PERSON
Wembley Stadium           -> GPE
London                    -> GPE
Cristiano Ronaldo         -> PERSON
Marcus Rashford           -> PERSON
July 2024                 -> DATE


In [ ]:
# Word embeddings with gensim Word2Vec
# Setup: python -m pip install gensim nltk
!pip install gensim nltk

import nltk
nltk.download("punkt")
nltk.download("punkt_tab") # Added to resolve LookupError
from nltk.tokenize import sent_tokenize, word_tokenize

from gensim.models import Word2Vec

corpus = """
Manufacturing relies on predictive maintenance and supply chain optimization.
Data engineers build pipelines, while analysts monitor KPIs and anomalies.
Robotics and IoT sensors stream telemetry to cloud databases for real-time insights.
Quality control uses computer vision to detect defects on the shop floor.
"""

# Tokenize sentences -> words
sentences = [word_tokenize(s.lower()) for s in sent_tokenize(corpus)]

# Train a small Word2Vec model
model = Word2Vec(
    sentences,
    vector_size=50,   # embedding dimension
    window=5,         # context window size
    min_count=1,      # keep all words for demo
    workers=2,
    sg=1              # skip-gram; use 0 for CBOW
)

# Explore similar words
for target in ["maintenance", "supply", "quality", "telemetry"]:
    print(f"\nTop similar to '{target}':")
    try:
        for w, score in model.wv.most_similar(target, topn=5):
            print(f"{w:15} -> {score:.3f}")
    except KeyError:
        print("Word not in vocabulary.")

# Cosine similarity between pairs
from numpy import dot
from numpy.linalg import norm

def cosine(u, v):
    return dot(u, v) / (norm(u) * norm(v))

pairs = [("maintenance", "telemetry"),
         ("quality", "defects"),
         ("supply", "optimization")]

print("\nCosine similarities:")
for a, b in pairs:
    try:
        sim = cosine(model.wv[a], model.wv[b])
        print(f"{a:12} ~ {b:12} -> {sim:.3f}")
    except KeyError:
        print(f"Missing word: {a} or {b}")


Top similar to 'maintenance':
real-time       -> 0.240
defects         -> 0.240
analysts        -> 0.236
kpis            -> 0.224
detect          -> 0.204

Top similar to 'supply':
optimization    -> 0.286
to              -> 0.252
cloud           -> 0.192
analysts        -> 0.169
maintenance     -> 0.169

Top similar to 'quality':
shop            -> 0.307
cloud           -> 0.267
uses            -> 0.191
floor           -> 0.154
.               -> 0.151

Top similar to 'telemetry':
predictive      -> 0.171
cloud           -> 0.171
manufacturing   -> 0.167
iot             -> 0.166
for             -> 0.164

Cosine similarities:
maintenance  ~ telemetry    -> -0.145
quality      ~ defects      -> 0.119
supply       ~ optimization -> 0.286


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
from collections import defaultdict

# Group entities by their label
by_label = defaultdict(list)
for ent in doc.ents:
    by_label[ent.label_].append(ent.text)

print("\nEntities grouped by label:")
for label, items in by_label.items():
    print(f"{label}: {sorted(set(items))}")


Entities grouped by label:
PERSON: ['Cristiano Ronaldo', 'Manchester United', 'Marcus Rashford']
GPE: ['London', 'Wembley Stadium']
DATE: ['July 2024']


In [ ]:
# Word embeddings with gensim Word2Vec for healthcare domain
!pip install gensim nltk

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize, word_tokenize

from gensim.models import Word2Vec

corpus = """
Patients with diabetes require careful diagnosis and ongoing treatment plans.
Symptoms such as fatigue and blurred vision are common indicators.
Medication adherence is crucial for managing chronic conditions.
Doctors and nurses collaborate to provide comprehensive patient care.
Research focuses on new therapies and early detection methods.
"""

# Tokenize sentences -> words
sentences = [word_tokenize(s.lower()) for s in sent_tokenize(corpus)]

# Train a small Word2Vec model
model = Word2Vec(
    sentences,
    vector_size=50,
    window=5,
    min_count=1,
    workers=2,
    sg=1
)

# Explore similar words relevant to healthcare
for target in ["diagnosis", "treatment", "symptoms", "medication", "patients"]:
    print(f"\nTop similar to '{target}':")
    try:
        for w, score in model.wv.most_similar(target, topn=5):
            print(f"{w:15} -> {score:.3f}")
    except KeyError:
        print("Word not in vocabulary.")

# Cosine similarity between healthcare-related pairs
from numpy import dot
from numpy.linalg import norm

def cosine(u, v):
    return dot(u, v) / (norm(u) * norm(v))

pairs = [("diagnosis", "symptoms"),
         ("treatment", "medication"),
         ("doctors", "nurses"),
         ("patients", "care")]

print("\nCosine similarities:")
for a, b in pairs:
    try:
        sim = cosine(model.wv[a], model.wv[b])
        print(f"{a:12} ~ {b:12} -> {sim:.3f}")
    except KeyError:
        print(f"Missing word: {a} or {b}")


Top similar to 'diagnosis':
to              -> 0.430
patient         -> 0.290
with            -> 0.235
diabetes        -> 0.197
managing        -> 0.182

Top similar to 'treatment':
symptoms        -> 0.286
methods         -> 0.251
conditions      -> 0.191
common          -> 0.169
ongoing         -> 0.168

Top similar to 'symptoms':
treatment       -> 0.286
vision          -> 0.270
plans           -> 0.214
are             -> 0.179
to              -> 0.166

Top similar to 'medication':
doctors         -> 0.278
ongoing         -> 0.224
and             -> 0.222
therapies       -> 0.215
research        -> 0.201

Top similar to 'patients':
.               -> 0.254
research        -> 0.254
conditions      -> 0.170
to              -> 0.158
methods         -> 0.151

Cosine similarities:
diagnosis    ~ symptoms     -> 0.148
treatment    ~ medication   -> -0.144
doctors      ~ nurses       -> 0.058
patients     ~ care         -> 0.066


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
!pip install transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Choose a pre-trained model. 'gpt2' is a good starting point for demonstrations.
# For a more robust chatbot, you might consider larger models or fine-tuning.
model_name = "gpt2"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the language model
model = AutoModelForCausalLM.from_pretrained(model_name)

print(f"Tokenizer and model '{model_name}' loaded successfully.")
print("Model architecture:")
print(model)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Tokenizer and model 'gpt2' loaded successfully.
Model architecture:
GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257,

In [ ]:
# Example student query
student_query = "What are the admission requirements for the Computer Science program?"

# Tokenize the input query
inputs = tokenizer(student_query, return_tensors="pt", max_length=1024, truncation=True)

print("Original Query:", student_query)
print("\nTokenized IDs (input_ids):")
print(inputs["input_ids"])
print("\nDecoded Tokens:")
print(tokenizer.decode(inputs["input_ids"][0]))
print("\nAttention Mask:")
print(inputs["attention_mask"])

Original Query: What are the admission requirements for the Computer Science program?

Tokenized IDs (input_ids):
tensor([[ 2061,   389,   262, 13938,  5359,   329,   262, 13851,  5800,  1430,
            30]])

Decoded Tokens:
What are the admission requirements for the Computer Science program?

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [ ]:
# Generate a response using the model
output_sequences = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"], # Added attention mask
    pad_token_id=tokenizer.eos_token_id,    # Added pad token ID
    max_length=100, # Max length of the generated response
    num_return_sequences=1,
    no_repeat_ngram_size=2, # Avoid repetitive phrases
    do_sample=True, # Enable sampling for more creative responses
    top_k=50, # Consider top 50 most likely words
    top_p=0.95, # Nucleus sampling
    temperature=0.7, # Controls randomness: lower is more deterministic
    eos_token_id=tokenizer.eos_token_id # End of sequence token
)

# Decode the generated sequence
generated_text = tokenizer.decode(output_sequences[0], skip_special_tokens=True)

print("Generated Response:")
print(generated_text)

Generated Response:
What are the admission requirements for the Computer Science program?

The Computer History program is designed to provide students with a strong understanding of computer science in the context of its current state. It consists of six courses: Computer Systems Analysis, Computer Software Engineering, Electrical Engineering and Computer Architecture. The course covers a wide range of topics in Computer Theory and Applications, as well as Computer Security, Machine Learning, and Programming Languages.
"The best Computer science program in Oregon is Computer Biology. We have a


In [ ]:
# A more engineered prompt for the helpdesk chatbot
engineered_prompt = (
    "You are a helpful university helpdesk assistant. Your task is to provide accurate and concise information "
    "to students. Respond to the following query:\n\n"
    "Student: What is the last date to apply for the scholarship program for international students?"
    "\nAssistant:"
)

# Tokenize the engineered prompt
inputs_engineered = tokenizer(engineered_prompt, return_tensors="pt", max_length=1024, truncation=True)

# Generate a response with the engineered prompt
output_sequences_engineered = model.generate(
    inputs_engineered["input_ids"],
    attention_mask=inputs_engineered["attention_mask"], # Added attention mask
    pad_token_id=tokenizer.eos_token_id,                # Added pad token ID
    max_length=200, # Increased max_length for potentially longer responses
    num_return_sequences=1,
    no_repeat_ngram_size=2,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.7,
    eos_token_id=tokenizer.eos_token_id
)

# Decode and print the generated response
generated_text_engineered = tokenizer.decode(output_sequences_engineered[0], skip_special_tokens=True)

print("Engineered Prompt:")
print(engineered_prompt)
print("\nGenerated Response with Engineered Prompt:")
print(generated_text_engineered)

Engineered Prompt:
You are a helpful university helpdesk assistant. Your task is to provide accurate and concise information to students. Respond to the following query:

Student: What is the last date to apply for the scholarship program for international students?
Assistant:

Generated Response with Engineered Prompt:
You are a helpful university helpdesk assistant. Your task is to provide accurate and concise information to students. Respond to the following query:

Student: What is the last date to apply for the scholarship program for international students?
Assistant: June
I think it's been a couple of months.
Thank you.


In [ ]:
# STEP 3: Define University Knowledge Base
# ------------------------------------------
knowledge_base = """
University Helpdesk Information

Hostel Admission Requirements:
- Admission confirmation letter
- Government ID proof
- Two passport size photographs

Course Registration:
- Registration happens online through the university portal
- Students must register before the semester deadline

Fee Payment:
- Fees can be paid through the student dashboard
- Online payment methods include debit card, credit card, and net banking

Examination Schedule:
- Semester exams usually start in December and May

Library Access:
- Students must carry their university ID card
"""

In [ ]:
import re

def retrieve_info(query, knowledge_base):
    sections = []
    current_section = None
    current_content = []

    # Split knowledge base into lines and process
    for line in knowledge_base.strip().split('\n'):
        line = line.strip()
        if not line:
            continue

        # Check if it's a new section header (e.g., "Hostel Admission Requirements:")
        if line.endswith(':'):
            if current_section:
                sections.append({
                    'header': current_section,
                    'content': '\n'.join(current_content)
                })
            current_section = line[:-1].strip()
            current_content = []
        else:
            # Add bullet points or other content to the current section
            current_content.append(line)

    # Add the last section
    if current_section:
        sections.append({
            'header': current_section,
            'content': '\n'.join(current_content)
        })

    # Extract keywords from the query
    query_keywords = set(word.lower() for word in re.findall(r'\b\w+\b', query))

    # Calculate relevance scores
    relevance_scores = []
    for i, section in enumerate(sections):
        score = 0
        section_text = (section['header'] + ' ' + section['content']).lower()
        for keyword in query_keywords:
            if keyword in section_text:
                score += 1
        relevance_scores.append((score, i))

    # Find the maximum score
    if not relevance_scores:
        return "No relevant information found."

    max_score = 0
    if relevance_scores:
        max_score = max(s for s, _ in relevance_scores)

    if max_score == 0:
        return "No relevant information found based on your query keywords."

    # Get all sections with the maximum score
    most_relevant_sections = []
    for score, i in relevance_scores:
        if score == max_score:
            most_relevant_sections.append(sections[i])

    # Format the output
    response_parts = []
    for section in most_relevant_sections:
        response_parts.append(f"--- {section['header']} ---\n{section['content']}")

    return "\n\n".join(response_parts)

print("retrieve_info function defined.")

retrieve_info function defined.


In [ ]:
student_query_rag = "What are the requirements for hostel admission?"
retrieved_context = retrieve_info(student_query_rag, knowledge_base)

print("Student Query:", student_query_rag)
print("\nRetrieved Context:")
print(retrieved_context)

Student Query: What are the requirements for hostel admission?

Retrieved Context:
--- Hostel Admission Requirements ---
- Admission confirmation letter
- Government ID proof
- Two passport size photographs


In [ ]:
def generate_rag_response(student_query, knowledge_base, tokenizer, model, max_length=200):
    # 1. Retrieve relevant information
    retrieved_context = retrieve_info(student_query, knowledge_base)

    # 2. Augment the prompt
    if "No relevant information" in retrieved_context:
        # If no relevant context is found, use a general prompt
        augmented_prompt = (
            f"You are a helpful university helpdesk assistant. Respond to the following query. "
            f"If you don't know the answer, state that you don't have information on this topic. "
            f"Student: {student_query}\nAssistant:"
        )
    else:
        # Augment the prompt with retrieved context
        augmented_prompt = (
            f"You are a helpful university helpdesk assistant. Use the following information to answer the student's query. "
            f"If the answer is not in the provided information, state that you don't have information on this topic. "
            f"\n\nKnowledge Base:\n{retrieved_context}\n\nStudent: {student_query}\nAssistant:"
        )

    # 3. Tokenize the augmented prompt
    inputs = tokenizer(augmented_prompt, return_tensors="pt", max_length=1024, truncation=True)

    # 4. Generate a response using the LLM
    output_sequences = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        pad_token_id=tokenizer.eos_token_id,
        max_length=max_length,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id
    )

    # 5. Decode the generated sequence
    generated_text = tokenizer.decode(output_sequences[0], skip_special_tokens=True)

    # Remove the prompt from the generated text to get only the assistant's response
    response_start_index = generated_text.find("Assistant:")
    if response_start_index != -1:
        return generated_text[response_start_index + len("Assistant:"):].strip()
    else:
        return generated_text.strip()

print("generate_rag_response function defined.")

generate_rag_response function defined.


In [ ]:
student_query_rag = "What are the requirements for hostel admission?"
rag_response = generate_rag_response(student_query_rag, knowledge_base, tokenizer, model)

print("Student Query:", student_query_rag)
print("\nRAG Assistant Response:")
print(rag_response)

Student Query: What are the requirements for hostel admission?

RAG Assistant Response:
Your university will require you to take the "Student's Information" online. You will need to complete the information before you can enter the hostels.
. Please note that if you are unsure about the details of the admission you must contact the University's admissions office by email. A full list of information about your university can be found on the university's website. In addition, you may be asked to fill out an online application to be considered for admission. Your application must include all of: (1) the name, address, telephone number, email address


In [ ]:
# ==========================================
# University Helpdesk Chatbot (Complete Code)
# ==========================================

# STEP 1: Import Libraries
import re

# STEP 2: Define University Knowledge Base
knowledge_base = {
    "hostel admission": [
        "Admission confirmation letter",
        "Government ID proof",
        "Two passport size photographs"
    ],

    "course registration": [
        "Registration happens online through the university portal",
        "Students must register before the semester deadline"
    ],

    "fee payment": [
        "Fees can be paid through the student dashboard",
        "Online payment methods include debit card, credit card, and net banking"
    ],

    "examination schedule": [
        "Semester exams usually start in December and May"
    ],

    "library access": [
        "Students must carry their university ID card"
    ]
}

# STEP 3: Function to Search Knowledge Base
def get_answer(question):

    question = question.lower()

    for topic, info in knowledge_base.items():
        if topic in question:
            return "\n".join(info)

    return "Sorry, I don't have information about that. Please contact the university helpdesk."

# STEP 4: Chatbot Interface
def university_chatbot():

    print("🎓 University Helpdesk Chatbot")
    print("Type 'exit' to stop the chatbot\n")

    while True:

        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("Chatbot: Thank you! Have a great day.")
            break

        response = get_answer(user_input)
        print("Chatbot:", response)
        print()

# STEP 5: Run Chatbot
university_chatbot()

🎓 University Helpdesk Chatbot
Type 'exit' to stop the chatbot

Chatbot: Admission confirmation letter
Government ID proof
Two passport size photographs

